# CS 228 data build (run once, CPU is enough)

Builds `data_pad2` (face crops, many people, max 8 photos per person per class) and keeps it in this notebook's
Output, so the experiment notebook can reuse it instead of rebuilding for 30+ minutes.

**Use:** Add Input -> Datasets -> `celeba-spoof-for-face-antispoofing` (attentionlayer241). Session options:
Accelerator = **None (CPU)**, Internet = On. Then **Save Version -> Save & Run All (Commit)**. Takes about 40 min.
The build is deterministic: the same command always produces the same images.

In [ ]:
# 1. Get the code
%cd /kaggle/working
!rm -rf cs228-pad-project
GITHUB_USER = "ShivamShelar101"
!git clone https://github.com/{GITHUB_USER}/cs228-pad-project.git
%cd cs228-pad-project
!pip install -q onnxscript matplotlib
!git log --oneline -1

In [ ]:
# 2. Dataset location (attach it first: Add Input -> Datasets -> celeba-spoof-for-face-antispoofing)
import glob, os
hits = []
for depth in range(1, 4):   # shallow search only: never walk the 750k-file dataset
    hits += glob.glob("/kaggle/input/" + "*/" * (depth - 1) + "celeba-spoof-for-face-antispoofing")
print("found:", hits)
assert hits, "Dataset not attached: Add Input -> Datasets -> celeba-spoof-for-face-antispoofing"
SRC = hits[0]
print("dataset at:", SRC)

In [ ]:
# 3. Build the face crops (prints progress every 500 crops)
!python -m data.prepare_celeba_spoof --src {SRC} --dst ./data_pad2 --per_class 3000 --test_per_class 600 --max_per_subject 8
!for s in train val test; do for c in live spoof; do echo $s/$c $(ls data_pad2/$s/$c | wc -l); done; done

In [ ]:
# 4. Keep only the data in the output (smaller, faster to attach)
!rm -rf .git checkpoints results
!du -sh data_pad2